# Nexus / GAIA — Gemini Colab Backend\nInstalls the Colab-side Gemini REST backend used by the canonical `Host colab` SSH transport. The credential is read from the Colab secret `GEMINI_API_KEY`; it is not stored in this notebook.\n

In [ ]:
import os\ntry:\n    from google.colab import userdata\n    if not os.environ.get('GEMINI_API_KEY'):\n        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') or ''\nexcept Exception:\n    pass\nprint('GEMINI_API_KEY configured:', bool(os.environ.get('GEMINI_API_KEY')))\n

In [ ]:
from pathlib import Path\nbackend = r'''#!/usr/bin/env python3\nimport json, os, sys, urllib.request, urllib.error\nkey=os.environ.get('GEMINI_API_KEY','').strip()\nmodel=os.environ.get('GEMINI_MODEL','gemini-flash-latest')\nif not key:\n    print(json.dumps({'error':'GEMINI_API_KEY is not configured'})); raise SystemExit(78)\nprompt=sys.stdin.read().strip() if not sys.argv[1:] else ' '.join(sys.argv[1:]).strip()\nif not prompt:\n    print(json.dumps({'error':'empty_prompt'})); raise SystemExit(64)\nurl=f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent'\nbody=json.dumps({'contents':[{'parts':[{'text':prompt}]}]}).encode()\nreq=urllib.request.Request(url,data=body,method='POST',headers={'Content-Type':'application/json','X-goog-api-key':key})\ntry:\n    with urllib.request.urlopen(req,timeout=120) as response: sys.stdout.write(response.read().decode())\nexcept urllib.error.HTTPError as e:\n    sys.stderr.write(e.read().decode(errors='replace')); raise SystemExit(70)\n'''\ntarget=Path('/root/nexus/bin/gemini-backend')\ntarget.parent.mkdir(parents=True, exist_ok=True)\ntarget.write_text(backend)\ntarget.chmod(0o700)\nprint('installed:', target)\n

In [ ]:
# Real smoke test; runs only when this cell is executed.\nimport subprocess, json\np=subprocess.run(['/root/nexus/bin/gemini-backend'],input='Return exactly: NEXUS_COLAB_GEMINI_OK',text=True,capture_output=True)\nprint('exit:',p.returncode)\nif p.stdout:\n    data=json.loads(p.stdout)\n    print('response:',data.get('candidates',[{}])[0].get('content',{}).get('parts',[{}])[0].get('text','').strip())\nif p.stderr: print('stderr:',p.stderr[:1000])\n

## Client-side SSH entry\nConfigure the calling Nexus/SYNAPSE node with `Host colab`, `ProxyCommand colab ssh --proxy-mode -s colab`, `RequestTTY no`, and remote command `/root/nexus/bin/gemini-backend`. Prompts travel over stdin and the provider adapter hashes the raw JSON response for the ADAM receipt.\n